# 2. FranceCrops cabbage and satellite imagery

Load verified FranceCrops cabbage training footprints, select **7,000 candidate pixels**, and retrieve 2022 Sentinel-2 imagery and actual-date summaries. Then use [Four-month windows](Crop_Four_Month_Windows.ipynb) to retain up to **500 usable cabbage pixels**.

Run cells in order. Network access is needed for source tables and Earth Engine retrieval. Reruns reuse verified files. This notebook never downloads the large interpolated FranceCrops `x` arrays: geometry-based retrieval keeps the imagery rules consistent with Sen4AgriNet.

[Dataset card](https://huggingface.co/datasets/saget-antoine/francecrops) · [Geometry schema](https://huggingface.co/datasets/saget-antoine/francecrops/blob/main/data/geolocation/README.md)

Source labels are administrative declarations, not independently verified crop-cycle dates. Only FranceCrops `train` parcels are used. The repository revision is pinned for reproducibility.



**Expanded timing-first bilinear run:** 7,000 cabbage candidate pixels. Daily summaries, satellite downloads, and Earth Engine sign-in are enabled for this rerun. Run sequentially: offline capacity and sampling → daily summaries → timing gate → qualified-window imagery → handoff. Existing pilot files remain in their original folders.

**Minimum acquisition pool:** each crop must have more than 3,000 unique sampled pixels remaining after timing filtering before imagery downloading starts. Repeated half-month images and rectangular context pixels do not increase this count. The final usable target remains 500 per crop; incomplete pixels remain excluded.


## 1. Set local paths and pilot settings

Install the repository requirements in your notebook kernel first.

Output: `Source tables / Pilot fields / Satellite imagery: <path>` — separate FranceCrops folders.

Output: `Candidate target: 7000` — pixels before quality filtering; `Usable target: 500` — final cap in the next notebook.


In [ ]:
from pathlib import Path
import json
import os
import francecrops_workflow
import pilot_sampling
from crop_workflow import save_json

# Collection is enabled for this rerun; successful downloads remain reusable.
import bilinear_collection
run_name = bilinear_collection.COLLECTION_NAME
retrieve_daily_summaries = True
download_satellite_imagery = True

settings_path = Path("settings.local.json")
local_settings = json.loads(settings_path.read_text()) if settings_path.exists() else {}
report_folder = Path(local_settings.get("report_root", "./sen4agrinet_work/inventory")).resolve()
france_root = report_folder.parent / "francecrops"
revision = francecrops_workflow.DEFAULT_REVISION
source_table_folder = france_root / "source_metadata" / revision
field_folder = france_root / "fields" / "2022" / run_name
l2a_reports = france_root / "l2a" / "2022" / run_name
dataset_folder = (
    Path(os.environ.get("S4A_ROOT", local_settings.get("dataset_root", "./S4A")))
    .expanduser()
    .resolve()
)
l2a_folder = (
    Path(
        local_settings.get(
            "francecrops_imagery_root", str(dataset_folder / "sentinel2_francecrops")
        )
    )
    .expanduser()
    .resolve()
    / run_name
)
timeline_folder = france_root / "analysis" / "2022" / run_name / "timelines"
earth_engine_project = os.environ.get(
    "EE_PROJECT", local_settings.get("earth_engine_project", "YOUR_EARTH_ENGINE_PROJECT")
)
candidate_pixels_per_crop = bilinear_collection.CANDIDATE_BUDGETS["cabbage"]
usable_pixels_per_crop = pilot_sampling.USABLE_PIXELS_PER_CROP
sampling_seed = 42
cloud_probability_threshold = 40
minimum_valid_fraction = 0.8
authenticate_now = True
refresh_inventory = False
print("Source tables:", source_table_folder)
print("Pilot fields:", field_folder)
print("Satellite imagery:", l2a_folder)
print("Candidate target:", candidate_pixels_per_crop)
print("Usable target:", usable_pixels_per_crop)


## 2. Load cached labels and parcel footprints

Use the reference tables already downloaded by the earlier run. Missing tables stop preparation with an error; this cell never downloads files.

Output: `Reference tables: <paths>` identifies the cached label and geometry inputs. The large interpolated satellite arrays are not used.


In [ ]:
# Cached reference tables keep the offline preparation stage free of downloads.
table_paths = {name: source_table_folder / filename for name, filename in francecrops_workflow.TABLE_FILES.items()}
if any(not path.is_file() for path in table_paths.values()):
    raise FileNotFoundError("Restore cached FranceCrops reference tables before preparing this collection.")
print("Reference tables:", {name: str(path) for name, path in table_paths.items()})


## 3. Verify cabbage labels and select eligible parcels

Join `CHU` cabbage labels from the training split to WGS84 footprints. Invalid footprints are excluded and audited rather than silently repaired.

Output: `Valid cabbage training parcels: <number>` — parcel count, not pixel count.

Output: `Geometry audit: <path>` — original count and excluded footprints with reasons.


In [ ]:
geometry_audit_path = field_folder / "geometry_audit.json"
cabbage_features = francecrops_workflow.load_cabbage_parcels(
    table_paths, revision, geometry_audit_path
)
save_json(
    field_folder / "cabbage_training_parcels.geojson",
    {"type": "FeatureCollection", "features": cabbage_features},
)
print("Valid cabbage training parcels:", len(cabbage_features))
print("Geometry audit:", geometry_audit_path)


## 4. Select 7,000 unique candidate pixels

Use a 10 m inward parcel buffer to reduce boundary mixing. Prefer 60 spatially spread groups and add groups when small parcels cannot supply the budget, with at most 100 candidate pixels per group. Too little capacity raises an error; pixels are never duplicated to meet the target.

Output: `Interior candidate capacity: <number>` — capped candidate pool before field selection.

Output: `Selected cabbage pixels: 7000` — candidate count before cloud and window filtering.

Output: `Sampling summary: <summary>` — selected groups and candidate counts.


In [ ]:
cabbage_catalog = francecrops_workflow.build_cabbage_catalog(cabbage_features, sampling_seed)
candidate_budgets = {"cabbage": candidate_pixels_per_crop}
print("Reference capacity:", bilinear_collection.capacity_report(cabbage_catalog, candidate_budgets))
fragments, sampling_summary = bilinear_collection.prepare_candidates(
    cabbage_catalog, candidate_budgets, field_folder, sampling_seed
)
from season_workflow import file_sha256

save_json(
    field_folder / "source_provenance.json",
    {
        "dataset": francecrops_workflow.DATASET_NAME,
        "revision": revision,
        "year": 2022,
        "class_code": "CHU",
        "source_split": "train",
        "label_status": "reported",
        "interpolated_arrays_used": False,
        "license": "CC-BY-4.0",
        "reference_tables": {
            name: {"path": str(path), "sha256": file_sha256(path)}
            for name, path in table_paths.items()
        },
    },
)
print("Interior candidate capacity:", len(cabbage_catalog))
print("Selected cabbage pixels:", sampling_summary["candidate_pixels_per_crop"]["cabbage"])
print("Sampling summary:", sampling_summary)


## 5. Connect to Earth Engine

Set `authenticate_now = True` in Step 1 only when a new login is needed. Credentials are managed outside the notebook.

Output: `Earth Engine project: <value>` — registered project used for requests.

Output: `Imagery request version: 4` and `Resampling: <description>` — identifies the bilinear source processing needed by the new experiments.


### Configure trusted HTTPS certificates

Run this cell before Earth Engine authentication. It supplies the installed `certifi` CA bundle when Python has no explicit certificate-file setting, while preserving custom certificate settings. HTTPS certificate verification remains enabled.

Output: `HTTPS certificate bundle: <path>` — the trusted certificate file used by Python HTTPS connections.

Output: `Trusted CA certificates: <number>` — certificates loaded locally; this does not test a server connection or download imagery.


In [ ]:
import os
import ssl
import certifi

# Python.org installations may lack a default CA bundle; retain custom trust settings.
os.environ.setdefault("SSL_CERT_FILE", certifi.where())
certificate_context = ssl.create_default_context()
print("HTTPS certificate bundle:", os.environ["SSL_CERT_FILE"])
print("Trusted CA certificates:", certificate_context.cert_store_stats()["x509_ca"])


In [ ]:
from satellite_workflow import SatelliteRetriever, REQUEST_VERSION

if retrieve_daily_summaries:
    import ee
    if authenticate_now:
        ee.Authenticate()
    ee.Initialize(project=earth_engine_project)
    retriever = SatelliteRetriever(ee, 2022, l2a_folder, l2a_reports, cloud_probability_threshold)
else:
    print("Earth Engine connection skipped: retrieval is disabled.")
print("Imagery request version:", REQUEST_VERSION)
print("Resampling: B5/B11 bilinear before compositing; quality masks nearest-neighbor")


## 6. Inventory acquisitions and plan composites

Retrieve full-year 2022 imagery so the original interpolated February–November array coverage does not constrain detection. Annual crop labels still do not prove which crop occupied every date.

Output: `Inventoried regions: <number>` — successful parcel inventories.

Output: `Planned composite files: <number>` — 24 half-month files per successful region.

Output: `Inventory errors: <number>` — failed regions retained in `composite_plan.json` for retry.


In [ ]:
acquisition_plans, inventory_errors = [], []
if retrieve_daily_summaries:
    acquisition_plans, inventory_errors = retriever.build_plans(fragments, refresh=refresh_inventory)
print("Inventoried regions:", len(acquisition_plans))
print("Inventory errors:", len(inventory_errors))


## 7. Retrieve actual-date vegetation summaries

Small tables of NDVI, NDRE and NDMI support observed timing detection; this does not download additional rasters. Missing daily support cannot be replaced with interpolated benchmark arrays.

Output: `<region> <number> observed daily summaries` — actual satellite dates summarized per selected region.

Output: `Observed summary rows: <number>` — combined region-date count. Failures are retained in `observed_errors.json`.


In [ ]:
observed_rows = []
if retrieve_daily_summaries:
    import observed_timelines
    observed_rows = observed_timelines.retrieve_observed_timelines(
        acquisition_plans, retriever.mask_acquisition, ee, timeline_folder, minimum_valid_fraction
    )
print("Observed summary rows:", len(observed_rows))


### 7.1 Timing gate: select windows before downloading imagery

Output: `Timing-qualified regions` and `Selected composite intervals` count the regions and eight-bin windows that passed the unchanged automatic policy. They are not usable pixel counts. Reports in `timing_first/` retain rejected outcomes and adjacent-year needs. No thresholds or completeness requirements are relaxed.


In [ ]:
selected_plans = []
if retrieve_daily_summaries:
    selected_plans = bilinear_collection.select_timing_plans(
        acquisition_plans, observed_rows, l2a_reports / "timing_first"
    )
    print("Timing-qualified regions:", len(selected_plans))
    print("Unique sampled pixels planned for download:", bilinear_collection.download_pixel_counts(selected_plans, candidate_budgets))
    print("Selected composite intervals:", sum(len(plan["download_intervals"]) for plan in selected_plans))
else:
    print("Timing gate skipped: daily-summary retrieval is disabled.")


## 8. Download timing-qualified satellite composites

SCL, exact cloud-probability matches and valid scene edges are applied before daily mosaics and half-month medians. These are the same rules used for Sen4AgriNet.

Output: `<region> <interval> <status>` — per-file progress.

Output: `Downloaded / Reused / Failed composites: <number>` — new files, checksum-verified existing files, and requests needing retry. Failed and empty intervals remain explicit.


In [ ]:
collection_ready = False
if download_satellite_imagery and not retrieve_daily_summaries:
    raise ValueError("Enable daily summaries and pass the timing gate before downloading imagery.")
if download_satellite_imagery:
    if not selected_plans:
        raise ValueError("No timing-qualified windows; no imagery will be downloaded.")
    download_pixels = bilinear_collection.require_download_pixel_budget(selected_plans, candidate_budgets)
    print("Verified download-pixel budget:", download_pixels)
    download_summary = retriever.download_composites(selected_plans)
    completed_budget = bilinear_collection.verify_download_completion(
        selected_plans, download_summary, candidate_budgets, l2a_reports / "download_pixel_budget.json"
    )
    collection_ready = True
    print("Downloaded sampled pixels:", completed_budget["downloaded_sampled_pixels"])
else:
    download_summary = {"downloaded": 0, "reused": 0, "failed": 0}
    print("Satellite downloads disabled.")
print("Downloaded composites:", download_summary["downloaded"])
print("Reused composites:", download_summary["reused"])
print("Failed composites:", download_summary["failed"])


## 9. Save the source handoff

Output: `FranceCrops source handoff: <path>` — paths and observation year loaded by the four-month notebook. Quality checks there determine actual usable counts.

Continue with [Four-month windows](Crop_Four_Month_Windows.ipynb). The combined prototype target is **500 usable pixels each for carrot, cabbage and potato**. A shortfall remains a shortfall; no samples are fabricated.


In [ ]:
from pilot_sources import save_source_manifest, save_json

source_handoff_path = report_folder.parent / "source_handoffs" / f"francecrops_{run_name}.json"
if collection_ready:
    source_manifest = save_source_manifest(
        source_handoff_path,
        "francecrops",
        2022,
        field_folder,
        l2a_folder,
        l2a_reports / "composite_progress",
        timeline_folder,
        revision,
    )
    print("FranceCrops source handoff:", source_handoff_path)
    source_manifest["candidate_target_per_crop"] = candidate_budgets
    save_json(source_handoff_path, source_manifest)
else:
    print("Source handoff not exported: the required download-pixel budget has not completed.")


## Continue to four-month windows

After both source downloads finish, open **Crop_Four_Month_Windows.ipynb**, set `source_run_name = "bilinear_expanded"` and `allow_existing_version_3 = False`, and run sequentially. The final cap stays at 500 usable pixels per crop. For future feature engineering and training, set `analysis_run_name = "bilinear_expanded"` in those notebooks. Current version 3 pilot outputs remain available in the model-training notebook’s local exploratory mode.
